# Bronze CSV ingestion
Ingest CSV files from the Fabric landing zone, profile them, and persist a bronze table.


In [ ]:

from utils.logging_utils import get_logger
from utils.schema import read_native_with_schema_inference
from utils.validation import run_data_quality_checks

logger = get_logger("bronze")
landing_zone_native = "abfss://landingzone@stdnalnzukdev01.dfs.core.windows.net/20260923-fi4-beaconwellbeingtrust-native"

csv_path = f"{landing_zone_native}/care_delivery/*.csv"
bronze_csv_df = read_native_with_schema_inference(
    spark,
    csv_path,
    "csv",
    {"header": True, "mode": "PERMISSIVE"},
)

quality_metrics = run_data_quality_checks(
    bronze_csv_df,
    ["Episode ID"],
    {"Episode ID": "string", "Visit Date": "date", "Region": "string", "Care Hours": "double"},
)
logger.info("CSV quality metrics: %s", quality_metrics)
bronze_csv_df.write.mode("overwrite").saveAsTable("bronze.care_delivery_csv")
display(bronze_csv_df)
